In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

In [ ]:
metrics_dir = PROJECT_ROOT / "results" / "pilot" / "metrics"
files = [
    metrics_dir / "chronos2_ETTh1.parquet",
    metrics_dir / "moirai2_ETTh1.parquet",
    metrics_dir / "timesfm3_ETTh1.parquet",
    metrics_dir / "seasonal_naive_ETTh1.parquet"]
frames = []
for path in files:
    frame = pd.read_parquet(path)
    # Older result files mix microsecond and nanosecond timestamps.
    frame["cutoff_timestamp"] = frame["cutoff_timestamp"].dt.as_unit("ns")
    frames.append(frame)
metrics = pd.concat(frames, ignore_index=True)
metrics.head()

In [4]:
metrics.shape

(240, 12)

In [7]:
summary = (metrics
    .groupby(["prediction_length", "model"])
    .agg(
        windows=("window_id", "count"),
        mean_mase=("mase", "mean"),
        median_mase=("mase", "median"),
        mean_mae=("mae", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_smape=("smape", "mean"),
        mean_runtime=("inference_time_seconds", "mean"))
    .reset_index())

summary.round(4)

,prediction_length,model,windows,mean_mase,median_mase,mean_mae,mean_rmse,mean_smape,mean_runtime
0,24,chronos2,20,0.7684,0.5626,0.9455,1.2080,10.7808,0.0531
1,24,moirai2,20,0.7169,0.5212,0.8878,1.1660,10.3205,0.0288
2,24,seasonal_naive,20,1.2484,1.0696,1.5490,1.8236,18.3619,0.0000
3,24,timesfm3,20,0.7716,0.6863,0.9489,1.2032,10.7229,0.4205
4,96,chronos2,20,1.1722,1.2382,1.9478,2.2946,21.5013,0.0423
5,96,moirai2,20,1.3402,1.2002,2.2457,2.6104,25.4285,0.0427
6,96,seasonal_naive,20,1.5652,1.4210,2.6243,3.0852,29.8944,0.0000
7,96,timesfm3,20,1.2302,1.1428,2.0541,2.4002,22.5355,0.3144
8,192,chronos2,20,1.4476,1.1538,2.4795,2.9636,44.1432,0.0407
9,192,moirai2,20,1.5242,1.4695,2.6027,3.0864,49.2037,0.0669


In [10]:
tsfm_models = ["chronos2", "moirai2", "timesfm3"]
tsfm_metrics = metrics[metrics["model"].isin(tsfm_models)]
mase_table = tsfm_metrics.pivot(
    index=["prediction_length", "window_id"],
    columns="model",
    values="mase")
mase_table["best_tsfm"] = mase_table.idxmin(axis=1)
mase_table

model                        chronos2   moirai2  timesfm3 best_tsfm
prediction_length window_id                                        
24                0          0.302008  0.379346  0.418249  chronos2
                  1          0.725469  0.688865  0.772441   moirai2
                  2          1.734458  1.762020  1.844116  chronos2
                  3          0.853528  0.583212  0.733799   moirai2
                  4          0.557674  0.489419  0.589102   moirai2
                  5          0.616770  0.553078  0.724938   moirai2
                  6          0.567623  0.392172  0.788199   moirai2
                  7          0.420152  0.281050  0.305396   moirai2
                  8          0.293693  0.332797  0.295289  chronos2
                  9          0.747224  1.019976  0.809545  chronos2
                  10         0.542299  0.572779  0.647646  chronos2
                  11         0.352038  0.328690  0.346243   moirai2
                  12         1.972576  2.044229  1.932511  timesfm3
                  13         1.305860  0.376957  1.135684   moirai2
                  14         0.424567  0.259610  0.329645   moirai2
                  15         1.857328  1.560940  1.790205   moirai2
                  16         0.512090  0.939655  0.326710  timesfm3
                  17         0.423300  0.326211  0.316422  timesfm3
                  18         0.784387  1.013860  0.797790  chronos2
                  19         0.375030  0.432283  0.527999  chronos2
96                0          1.313357  1.792155  1.777597  chronos2
                  1          1.313904  1.610834  1.222500  timesfm3
                  2          0.631281  0.823346  0.696297  chronos2
                  3          0.893082  0.913468  1.063191  chronos2
                  4          1.310628  1.406968  0.857354  timesfm3
                  5          1.477328  1.524494  1.766720  chronos2
                  6          2.176214  2.671701  1.724405  timesfm3
                  7          1.287426  1.129053  1.586701   moirai2
                  8          0.834095  0.883191  0.960894  chronos2
                  9          1.691125  1.661442  1.878428   moirai2
                  10         2.430114  2.403937  2.836656   moirai2
                  11         0.502697  0.987026  0.486655  timesfm3
                  12         0.235687  0.951696  0.660323  chronos2
                  13         0.337714  0.528501  0.296052  timesfm3
                  14         0.957395  0.546187  0.552348   moirai2
                  15         1.188876  1.271371  1.263152  chronos2
                  16         1.392045  1.461744  1.285572  timesfm3
                  17         0.611294  0.677139  0.686702  chronos2
                  18         2.345231  2.519053  2.465539  chronos2
                  19         0.514281  1.041141  0.537015  chronos2
192               0          1.062263  1.140044  1.131396  chronos2
                  1          2.859165  2.803774  3.594495   moirai2
                  2          0.809533  2.036932  1.054112  chronos2
                  3          0.856458  0.807362  0.832334   moirai2
                  4          1.043355  1.061428  0.990751  timesfm3
                  5          1.004920  1.142856  0.995676  timesfm3
                  6          1.800879  1.713115  1.817299   moirai2
                  7          1.588995  1.522303  1.514017  timesfm3
                  8          1.192817  1.161029  1.470974   moirai2
                  9          2.496069  1.692659  2.236434   moirai2
                  10         1.042853  1.263536  1.221532  chronos2
                  11         1.050568  1.265916  1.119343  chronos2
                  12         1.938519  2.090308  1.628273  timesfm3
                  13         1.571339  1.929351  1.378214  timesfm3
                  14         1.626244  1.701517  1.637810  chronos2
                  15         2.215816  2.181533  2.664805   moirai2
                  16         0.657851  0.830837  0.59831

In [13]:
winner_counts = (mase_table 
    .reset_index()
    .groupby(
        ["prediction_length", "best_tsfm"]
    )
    .size()
    .unstack(fill_value=0))
winner_counts

best_tsfm,chronos2,moirai2,timesfm3
prediction_length,,,
24,7,10,3
96,10,4,6
192,6,7,7


In [14]:
all_models = metrics.pivot(
    index=["prediction_length", "window_id"],
    columns="model",
    values="mase")
all_models["best_model"] = all_models.idxmin(axis=1)
all_models

model                        chronos2   moirai2  seasonal_naive  timesfm3  \
prediction_length window_id                                                 
24                0          0.302008  0.379346        0.888932  0.418249   
                  1          0.725469  0.688865        0.887000  0.772441   
                  2          1.734458  1.762020        2.534228  1.844116   
                  3          0.853528  0.583212        1.864404  0.733799   
                  4          0.557674  0.489419        1.065874  0.589102   
                  5          0.616770  0.553078        1.322687  0.724938   
                  6          0.567623  0.392172        0.777660  0.788199   
                  7          0.420152  0.281050        0.334523  0.305396   
                  8          0.293693  0.332797        0.590754  0.295289   
                  9          0.747224  1.019976        1.448387  0.809545   
                  10         0.542299  0.572779        0.878370  0.647646   
                  11         0.352038  0.328690        0.965839  0.346243   
                  12         1.972576  2.044229        1.641400  1.932511   
                  13         1.305860  0.376957        2.318681  1.135684   
                  14         0.424567  0.259610        1.073399  0.329645   
                  15         1.857328  1.560940        1.850857  1.790205   
                  16         0.512090  0.939655        1.675298  0.326710   
                  17         0.423300  0.326211        0.835397  0.316422   
                  18         0.784387  1.013860        1.332051  0.797790   
                  19         0.375030  0.432283        0.681922  0.527999   
96                0          1.313357  1.792155        2.094066  1.777597   
                  1          1.313904  1.610834        2.468979  1.222500   
                  2          0.631281  0.823346        1.066742  0.696297   
                  3          0.893082  0.913468        0.939391  1.063191   
                  4          1.310628  1.406968        1.968082  0.857354   
                  5          1.477328  1.524494        1.058285  1.766720   
                  6          2.176214  2.671701        1.981301  1.724405   
                  7          1.287426  1.129053        1.405429  1.586701   
                  8          0.834095  0.883191        0.929546  0.960894   
                  9          1.691125  1.661442        1.669685  1.878428   
                  10         2.430114  2.403937        3.027253  2.836656   
                  11         0.502697  0.987026        1.250217  0.486655   
                  12         0.235687  0.951696        0.908639  0.660323   
                  13         0.337714  0.528501        0.593381  0.296052   
                  14         0.957395  0.546187        0.639140  0.552348   
                  15         1.188876  1.271371        1.436660  1.263152   
                  16         1.392045  1.461744        2.328862  1.285572   
                  17         0.611294  0.677139        1.319821  0.686702   
                  18         2.345231  2.519053        2.150390  2.465539   
                  19         0.514281  1.041141        2.067790  0.537015   
192               0          1.062263  1.140044        1.481916  1.131396   
                  1          2.859165  2.803774        2.121947  3.594495   
                  2          0.809533  2.036932        0.810468  1.054112   
                  3          0.856458  0.807362        1.037175  0.832334   
                  4          1.043355  1.061428        1.209562  0.990751   
                  5          1.004920  1.142856        1.381566  0.995676   
                  6          1.800879  1.713115        2.419233  1.817299   
                  7          1.588995  1.522303        2.862503  1.514017   
                  8          1.192817  1.161029        1.348630  1.470974   
                  9          2.496069  1.692659        1.846738  2.236434

In [15]:
all_models["best_model"].value_counts()

best_model
chronos2          20
moirai2           20
timesfm3          15
seasonal_naive     5
Name: count, dtype: int64

In [16]:
def winner_margin(row):
    values = row[tsfm_models].sort_values()
    return values.iloc[1] - values.iloc[0]


mase_table["winner_margin"] = mase_table.apply(winner_margin, axis=1)
mase_table[["best_tsfm", "winner_margin"]]

model                       best_tsfm  winner_margin
prediction_length window_id                         
24                0          chronos2       0.077339
                  1           moirai2       0.036605
                  2          chronos2       0.027561
                  3           moirai2       0.150587
                  4           moirai2       0.068255
                  5           moirai2       0.063692
                  6           moirai2       0.175451
                  7           moirai2       0.024346
                  8          chronos2       0.001596
                  9          chronos2       0.062321
                  10         chronos2       0.030480
                  11          moirai2       0.017553
                  12         timesfm3       0.040064
                  13          moirai2       0.758728
                  14          moirai2       0.070035
                  15          moirai2       0.229265
                  16         timesfm3       0.185380
                  17         timesfm3       0.009789
                  18         chronos2       0.013404
                  19         chronos2       0.057252
96                0          chronos2       0.464241
                  1          timesfm3       0.091404
                  2          chronos2       0.065015
                  3          chronos2       0.020386
                  4          timesfm3       0.453274
                  5          chronos2       0.047166
                  6          timesfm3       0.451808
                  7           moirai2       0.158374
                  8          chronos2       0.049097
                  9           moirai2       0.029683
                  10          moirai2       0.026177
                  11         timesfm3       0.016042
                  12         chronos2       0.424636
                  13         timesfm3       0.041662
                  14          moirai2       0.006161
                  15         chronos2       0.074276
                  16         timesfm3       0.106473
                  17         chronos2       0.065845
                  18         chronos2       0.120309
                  19         chronos2       0.022734
192               0          chronos2       0.069133
                  1           moirai2       0.055391
                  2          chronos2       0.244580
                  3           moirai2       0.024972
                  4          timesfm3       0.052604
                  5          timesfm3       0.009244
                  6           moirai2       0.087764
                  7          timesfm3       0.008286
                  8           moirai2       0.031788
                  9           moirai2       0.543775
                  10         chronos2       0.178679
                  11         chronos2       0.068775
                  12         timesfm3       0.310246
                  13         timesfm3       0.193125
                  14         chronos2       0.011566
                  15          moirai2       0.034282
                  16         timesfm3       0.059534
                  17          moirai2       0.026772
                  18         timesfm3       0.051100
                  19         chronos2       0.029131

In [17]:
mase_table["winner_margin"].describe()

count    60.000000
mean      0.115420
std       0.153606
min       0.001596
25%       0.027364
50%       0.058393
75%       0.127878
max       0.758728
Name: winner_margin, dtype: float64

In [18]:
def oracle_regret(row, model):
    return row[model] - row[tsfm_models].min()


for model in tsfm_models:
    mase_table[f"{model}_regret"] = mase_table.apply(lambda row: oracle_regret(row, model), axis=1)

In [19]:
mase_table[[
    "chronos2_regret",
    "moirai2_regret",
    "timesfm3_regret"]].mean()

model
chronos2_regret    0.107330
moirai2_regret     0.171707
timesfm3_regret    0.140525
dtype: float64